In [1]:
label_map = {
    "NONE": 0,
    "Thermal": 1,
    "Power": 2,
    "Mechanical": 3,
    "Communication": 4,
    "Control": 5
}

The other models outputs will be funneled into XGboost so XGboost can make the final decision, the cel below this one generates fake data for the other models until I get input data

In [5]:
import numpy as np

# ----------------------------
# Configuration
# ----------------------------

NUM_SAMPLES = 10000

PER_ERR_DIM = 64
LATENT_DIM = 32
GMM_DIM = 8
LSTM_DIM = 64
PCMCI_DIM = 20

NUM_CLASSES = 6

np.random.seed(42)

# ----------------------------
# Allocate arrays
# ----------------------------

X = np.zeros((NUM_SAMPLES, 188), dtype=np.float32)
y = np.zeros(NUM_SAMPLES, dtype=np.int64)

# ----------------------------
# Generate data
# ----------------------------

for i in range(NUM_SAMPLES):

    label = np.random.randint(0, NUM_CLASSES)

    y[i] = label

    # ------------------------
    # Base random noise
    # ------------------------

    per_err = np.random.normal(0.2, 0.05, PER_ERR_DIM)

    latent = np.random.normal(0.0, 1.0, LATENT_DIM)

    gmm = np.random.normal(0.0, 1.0, GMM_DIM)

    lstm = np.random.normal(0.0, 1.0, LSTM_DIM)

    pcmci = np.random.normal(0.0, 1.0, PCMCI_DIM)

    # ------------------------
    # Inject subsystem-specific patterns
    # ------------------------

    if label == 1:          # THERMAL

        per_err[0:12] += 3.0
        gmm[0] += 5
        lstm[0:8] += 2
        pcmci[0:3] += 2

    elif label == 2:        # POWER

        per_err[12:24] += 3.0
        gmm[1] += 5
        lstm[8:16] += 2
        pcmci[3:6] += 2

    elif label == 3:        # MECHANICAL

        per_err[24:40] += 3.0
        gmm[2] += 5
        lstm[16:32] += 2
        pcmci[6:10] += 2

    elif label == 4:        # COMMUNICATION

        per_err[40:52] += 3.0
        gmm[3] += 5
        lstm[32:48] += 2
        pcmci[10:15] += 2

    elif label == 5:        # CONTROL

        per_err[52:64] += 3.0
        gmm[4] += 5
        lstm[48:64] += 2
        pcmci[15:20] += 2

    # NONE (label 0) remains close to normal

    fused = np.concatenate([
        per_err,
        latent,
        gmm,
        lstm,
        pcmci
    ])

    X[i] = fused

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

np.save("fused_features.npy", X)
np.save("labels.npy", y)

Feature shape: (10000, 188)
Label shape: (10000,)


In [6]:
# train_xgboost.py

import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [7]:
X = np.load("fused_features.npy")   # shape: [N, 188]
y = np.load("labels.npy")           # shape: [N]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=6,
    eval_metric="mlogloss",
    tree_method="hist"
)

In [10]:
model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=True
)

[0]	validation_0-mlogloss:1.64467
[1]	validation_0-mlogloss:1.51928
[2]	validation_0-mlogloss:1.40971
[3]	validation_0-mlogloss:1.31232
[4]	validation_0-mlogloss:1.22508
[5]	validation_0-mlogloss:1.14649
[6]	validation_0-mlogloss:1.07481
[7]	validation_0-mlogloss:1.00921
[8]	validation_0-mlogloss:0.94908
[9]	validation_0-mlogloss:0.89350
[10]	validation_0-mlogloss:0.84201
[11]	validation_0-mlogloss:0.79432
[12]	validation_0-mlogloss:0.74968
[13]	validation_0-mlogloss:0.70804
[14]	validation_0-mlogloss:0.66923
[15]	validation_0-mlogloss:0.63300
[16]	validation_0-mlogloss:0.59900
[17]	validation_0-mlogloss:0.56695
[18]	validation_0-mlogloss:0.53681
[19]	validation_0-mlogloss:0.50850
[20]	validation_0-mlogloss:0.48182
[21]	validation_0-mlogloss:0.45676
[22]	validation_0-mlogloss:0.43307
[23]	validation_0-mlogloss:0.41069
[24]	validation_0-mlogloss:0.38960
[25]	validation_0-mlogloss:0.36963
[26]	validation_0-mlogloss:0.35078
[27]	validation_0-mlogloss:0.33293
[28]	validation_0-mlogloss:0.3

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor

In [11]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       341
           1       1.00      1.00      1.00       322
           2       1.00      1.00      1.00       335
           3       1.00      1.00      1.00       341
           4       1.00      1.00      1.00       322
           5       1.00      1.00      1.00       339

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



In [12]:
joblib.dump(model, "xgb_subsystem_model.pkl")

['xgb_subsystem_model.pkl']

### use this in attribution head

This format will work with the model dimensions VVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVV

In [13]:
import joblib
import numpy as np

class SubsystemAttributionHead:

    def __init__(self, model_path="xgb_subsystem_model.pkl"):
        self.model = joblib.load(model_path)

    def predict(self, fused_features: np.ndarray):
        """
        fused_features: shape [B, 188]
        """
        probs = self.model.predict_proba(fused_features)
        return probs